<a href="https://colab.research.google.com/github/parsa-asgari/bahar-bootcamp/blob/main/bahar_pinecone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Practice: Setup Pinecone + use OpenRouter to generate Embeddings

## 1. Install python packages

In [ ]:
!pip install pinecone openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 19.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pinecone]


In [ ]:
# API KEYS
PINECONE_API_KEY=''
PINECONE_INDEX_NAME=''
OPENROUTER_API_KEY=''

## 2. Generate a text file

In [ ]:
document = '''لورم ایپسوم متن ساختگی با تولید سادگی نامفهوم از صنعت چاپ گیرنده خلق گیرنده منابع اهداف قانون برندینگ صداقت توانبخشی. پردازش تغییر توزیع نشر ارزیابانه شبکه پرستاری فرستنده پژوهش پیشرفت! تغییر فلسفه سیستم داروسازی منطق گونه مدرن نوسازی تکثر فلسفه آینده.

صنعت آگاهی آنلاین نوسازی همگرایی کربن فناوری فناوری دهنده یکپارچگی؛ استفاده خلاقیت درمان بیمه مکان آنلاین استاندارد! آبی رابط کیفیت بهبود الگوریتم بهبود موفقیت دشواری شبکه کربن.

توزیع کتابهای فضا تبلیغات پاک تعامل تجدیدپذیر مسئولیت پرستاری ارزش کسب‌وکار. اجتماعی اقتدار گفت‌وگو مشاوره بهبود بازار برنامه‌ریزی منابع فارسی بهداشت همگرایی استاندارد بیمه بهبود آموزش. کاربری مشارکت اینترنت کارایی نرم‌افزار مجله حروفچینی نامفهوم اقناع گسترش صداقت پاک دیجیتال استفاده آزادی. کارایی سازمان اعتماد خلق خلق. '''


with open('document.txt', 'w') as file:
  file.write(document)


## 3. Create a Schema and an index in Pinecone

In [ ]:
import os, time
from pinecone import Pinecone, SchemaBuilder

pc = Pinecone(api_key=PINECONE_API_KEY)

schema = (
    SchemaBuilder()
      .add_dense_vector_field(name="embedding", dimension=1024, metric="cosine")
      .build()
)
if not pc.indexes.exists(name=PINECONE_INDEX_NAME):
    pc.indexes.create(name=PINECONE_INDEX_NAME, schema=schema)

while not pc.indexes.describe(name=PINECONE_INDEX_NAME).status.ready:
    time.sleep(2)

index = pc.Index(name=PINECONE_INDEX_NAME)

## 4. Generate Embeddings from `document.txt` and put it into Pinecone

In [ ]:
import os
from pathlib import Path
from openai import OpenAI
from pinecone import Pinecone

# OpenRouter embedding client
embedding_client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
)

MODEL = "liquid/lfm-2.5-embedding-350m:free"

# Pinecone client
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(PINECONE_INDEX_NAME)

# Read and chunk the text
text = Path("document.txt").read_text(encoding="utf-8")

chunk_size = 512
overlap = 20
chunks = []
start = 0

while start < len(text):
    chunk = text[start:start + chunk_size].strip()

    if chunk:
        chunks.append(chunk)

    start += chunk_size - overlap

# Embed and upload
batch_size = 10

for start in range(0, len(chunks), batch_size):
    batch = chunks[start:start + batch_size]

    response = embedding_client.embeddings.create(
        model=MODEL,
        input=batch,
    )
    for i, item in enumerate(response.data):
      print("😉====> This iks the resulting vector: ", i,item.embedding, "\n")
    documents = [
        {
            "_id": f"{start + i}",
            "embedding": item.embedding,
            "text": batch[i],
            "source": "document.txt",
        }
        for i, item in enumerate(response.data)
    ]
    print(documents)
    result = index.documents.upsert(
        namespace="__default__",
        documents=documents,
    )

    print(
        f"Uploaded {min(start + len(batch), len(chunks))}"
        f"/{len(chunks)} chunks"
    )

print("Finished uploading documents.")


😉====> This iks the resulting vector:  0 [0.0005615586997009814, 0.019991489127278328, 0.04537394270300865, 0.0026814427692443132, -0.046272434294223785, -0.08670465648174286, -0.0020777671597898006, 0.02212541177868843, 0.05682973936200142, 0.041555341333150864, 0.0768212303519249, -0.017071383073925972, 0.07098101824522018, -0.005166339688003063, -0.001474091550335288, 0.03324427455663681, 0.009434185922145844, -0.01740831881761551, 0.06559005379676819, 0.016509825363755226, -0.01740831881761551, 0.014095122925937176, 0.04470007121562958, -0.040207602083683014, 0.003144728485494852, 0.011511952616274357, -0.022013099864125252, -0.005082106217741966, 0.014151278883218765, 0.05593124404549599, -0.028751803562045097, -0.05885135009884834, 0.004913638345897198, 0.002976261079311371, 0.012354291044175625, 0.05143877491354942, 0.01841912418603897, 0.03301965072751045, -0.036164380609989166, 0.01864374801516533, 0.0048294044099748135, 0.016846761107444763, -0.009658808819949627, -0.07322724